# Etapa 4 — Aprendizado Probabilístico e EM

1. MLE e MAP para θ = P(facção) de uma Bernoulli, por UF.
2. Naive Bayes do zero para o problema da Etapa 1.
3. **EM com variável oculta Z = regime do município** (3 componentes). O Cartografias classifica cada município em *sem registro*, *uma facção (domínio)* ou *duas ou mais (disputa)*. O EM tenta **recuperar esses regimes só a partir dos homicídios**, sem ver o rótulo.
4. Naive Bayes puro × Naive Bayes com Z.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from utils import montar_tabela, atributos_numericos_log, ATRIBUTOS_CAT, DESCRICAO, SEED
np.random.seed(SEED)
pd.set_option("display.width", 140)
d = montar_tabela()
print(d.shape)

(808, 115)


## 1. MLE e MAP (prior Beta[2,2])

$$\hat\theta_{MLE} = \frac{k}{n} \qquad \hat\theta_{MAP} = \frac{k+1}{n+2}$$

Com poucos municípios, o MLE pode dar valores extremos (100% no Acre). O prior "puxa" a estimativa para o centro.

In [2]:
def mle(k, n): return k / n
def map_beta(k, n, a=2, b=2): return (k + a - 1) / (n + a + b - 2)
linhas = [["Amazônia Legal", len(d), int(d["ALVO"].sum())]] + \
         [[uf, len(s), int(s["ALVO"].sum())] for uf, s in d.groupby("UF")]
t = pd.DataFrame(linhas, columns=["recorte", "n", "k com facção"])
t["θ MLE"] = mle(t["k com facção"], t["n"]); t["θ MAP Beta(2,2)"] = map_beta(t["k com facção"], t["n"])
t.sort_values("n").round(4)

,recorte,n,k com facção,θ MLE,"θ MAP Beta(2,2)"
8,RR,15,14,0.9333,0.8824
3,AP,16,5,0.3125,0.3333
1,AC,22,22,1.0000,0.9583
7,RO,52,26,0.5000,0.5000
2,AM,62,20,0.3226,0.3281
9,TO,139,9,0.0647,0.0709
5,MT,141,42,0.2979,0.3007
6,PA,144,73,0.5069,0.5068
4,MA,217,48,0.2212,0.2237
0,Amazônia Legal,808,259,0.3205,0.3210


In [3]:
from scipy.stats import beta
th = np.linspace(0, 1, 400)
plt.figure(figsize=(5.5, 3.3))
plt.plot(th, beta.pdf(th, 2, 2), label="prior Beta(2,2)")
plt.plot(th, beta.pdf(th, 2 + 22, 2 + 0), label="posterior AC (k=22, n=22)")
plt.plot(th, beta.pdf(th, 2 + 14, 2 + 1), label="posterior RR (k=14, n=15)")
plt.axvline(1.0, ls="--", c="gray", label="MLE AC = 1,0")
plt.legend(fontsize=8); plt.xlabel("θ"); plt.tight_layout(); plt.savefig("figuras/etapa4_mle_map.png", dpi=150)

## 2. Naive Bayes do zero (atributos categóricos, suavização de Laplace)

In [4]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from utils import tercis_do_treino

class NaiveBayesCategorico:
    def __init__(self, alfa=1.0, limiar=0.5):
        self.alfa, self.limiar = alfa, limiar

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes = np.unique(y)
        self.log_prior = {c: np.log((y == c).mean()) for c in self.classes}
        self.log_cond = {}
        for a in X.columns:
            valores = X[a].unique(); V = len(valores)
            for c in self.classes:
                col = X[a][y == c]
                for v in valores:
                    self.log_cond[(a, v, c)] = np.log(((col == v).sum() + self.alfa) / (len(col) + self.alfa * V))
        self.atributos = list(X.columns)
        return self

    def predict_proba1(self, X):
        out = []
        for _, r in X.iterrows():
            s = {c: self.log_prior[c] + sum(self.log_cond.get((a, r[a], c), np.log(1e-6)) for a in self.atributos)
                 for c in self.classes}
            m = max(s.values()); ex = {c: np.exp(v - m) for c, v in s.items()}
            out.append(ex[1] / sum(ex.values()))
        return np.array(out)

    def predict(self, X):
        return (self.predict_proba1(X) >= self.limiar).astype(int)

X = d[ATRIBUTOS_CAT]; y = d["ALVO"].values
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

def cv_nb(Xm, limiar=0.5):
    acc, bal, f1 = [], [], []
    for tr, te in cv.split(Xm, y):
        Xf = tercis_do_treino(d, Xm, tr)      # cortes de TXH e TXT só com o treino
        p = NaiveBayesCategorico(limiar=limiar).fit(Xf.iloc[tr], y[tr]).predict(Xf.iloc[te])
        acc.append(accuracy_score(y[te], p)); bal.append(balanced_accuracy_score(y[te], p)); f1.append(f1_score(y[te], p))
    return np.mean(acc), np.mean(bal), np.mean(f1)

a, b, f = cv_nb(X); print(f"NB próprio: acurácia={a:.3f}  balanceada={b:.3f}  F1={f:.3f}")
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
acc = []
for tr, te in cv.split(X, y):
    Xo = OrdinalEncoder().fit_transform(tercis_do_treino(d, X, tr))
    acc.append(accuracy_score(y[te], CategoricalNB(alpha=1.0).fit(Xo[tr], y[tr]).predict(Xo[te])))
print(f"sklearn CategoricalNB: acurácia={np.mean(acc):.3f}")

NB próprio: acurácia=0.772  balanceada=0.737  F1=0.642
sklearn CategoricalNB: acurácia=0.772


In [5]:
# Resultado de cada dobra (a célula acima mostra só a média das 5)
linhas = []
for i, (tr, te) in enumerate(cv.split(X, y), 1):
    Xf = tercis_do_treino(d, X, tr)
    p = NaiveBayesCategorico().fit(Xf.iloc[tr], y[tr]).predict(Xf.iloc[te])
    com, sem = y[te] == 1, y[te] == 0
    linhas.append([i, len(te), com.sum(), (p[com] == 1).mean(), (p[sem] == 0).mean(),
                   accuracy_score(y[te], p), balanced_accuracy_score(y[te], p)])
por_dobra = pd.DataFrame(linhas, columns=["dobra", "n teste", "com facção", "acerto com facção",
                                          "acerto sem facção", "acurácia", "acur. balanceada"])
print(por_dobra.round(3).to_string(index=False))
print(f"\nMédia: acurácia = {por_dobra['acurácia'].mean():.3f} | balanceada = {por_dobra['acur. balanceada'].mean():.3f} "
      f"(desvio {por_dobra['acur. balanceada'].std():.3f}, de {por_dobra['acur. balanceada'].min():.3f} a {por_dobra['acur. balanceada'].max():.3f})")

 dobra  n teste  com facção  acerto com facção  acerto sem facção  acurácia  acur. balanceada
     1      162          52              0.654              0.827     0.772             0.741
     2      162          52              0.673              0.855     0.796             0.764
     3      162          52              0.635              0.809     0.753             0.722
     4      161          52              0.519              0.817     0.720             0.668
     5      161          51              0.706              0.873     0.820             0.789

Média: acurácia = 0.772 | balanceada = 0.737 (desvio 0.046, de 0.668 a 0.789)


## 3. EM com variável oculta Z (regime do município, K = 3)

Observamos x = log(1 + homicídios 2021–23) e supomos 3 regimes ocultos, cada um com distribuição normal. Depois comparamos os grupos encontrados pelo EM com os regimes do Cartografias, que o EM **nunca viu**.

In [6]:
from em import MisturaGaussianaEM
x = np.log1p(d["homicidio_total_2021_23"].values)
em = MisturaGaussianaEM(K=3, tol=1e-4).fit(x)
print(f"Convergiu em {em.n_iter} iterações")
print(pd.DataFrame({"θ": em.theta, "μ": em.mu, "σ": np.sqrt(em.var), "homicídios típicos (2021–23)": np.expm1(em.mu)},
                   index=["Z=0", "Z=1", "Z=2"]).round(3))
d["Z"] = em.predizer(x)
ct = pd.crosstab(d["Z"], d["REGIME"].map({0: "sem registro", 1: "uma facção", 2: "disputa"}), normalize="index")
print("\nComposição de cada grupo Z segundo o Cartografias (linhas somam 1):")
print(ct.round(3))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(range(1, em.n_iter + 1), em.log_verossimilhanca, "o-", ms=3)
ax[0].set_xlabel("iteração"); ax[0].set_ylabel("log-verossimilhança"); ax[0].set_title("EM: log-verossimilhança")
ax[1].hist(x, bins=30, density=True, alpha=.4)
xx = np.linspace(x.min(), x.max(), 300)
for k in range(3):
    ax[1].plot(xx, em.theta[k] * np.exp(-0.5 * (xx - em.mu[k]) ** 2 / em.var[k]) / np.sqrt(2 * np.pi * em.var[k]), label=f"Z={k}")
ax[1].set_xlabel("log(1 + homicídios 2021–23)"); ax[1].legend(); ax[1].set_title("Mistura estimada (K=3)")
plt.tight_layout(); plt.savefig("figuras/etapa4_em.png", dpi=150)
print("\nLog-verossimilhança não decrescente:", bool(np.all(np.diff(em.log_verossimilhanca) >= -1e-9)))

Convergiu em 5 iterações
         θ      μ      σ  homicídios típicos (2021–23)
Z=0  0.311  1.521  0.962                         3.575
Z=1  0.363  2.321  1.066                         9.181
Z=2  0.326  3.181  1.271                        23.083

Composição de cada grupo Z segundo o Cartografias (linhas somam 1):
REGIME  disputa  sem registro  uma facção
Z                                        
0         0.017         0.887       0.096
1         0.057         0.724       0.219
2         0.288         0.342       0.370

Log-verossimilhança não decrescente: True


In [7]:
# Mesma análise usando a TAXA por 100 mil (controla o tamanho do município)
x_taxa = np.log1p(d["taxa_homicidio_2021_23"].values)
em_t = MisturaGaussianaEM(K=3).fit(x_taxa)
print(f"EM com a taxa: {em_t.n_iter} iterações | taxas típicas por 100 mil: {np.expm1(em_t.mu).round(1)}")
print(pd.crosstab(em_t.predizer(x_taxa), d["REGIME"].map({0: "sem registro", 1: "uma facção", 2: "disputa"}), normalize="index").round(3))

EM com a taxa: 38 iterações | taxas típicas por 100 mil: [ 0.  15.6 29.3]
REGIME  disputa  sem registro  uma facção
row_0                                    
0         0.000         0.942       0.058
1         0.061         0.720       0.219
2         0.152         0.616       0.232


**Contagem × taxa.** Os grupos do EM acompanham melhor os regimes do Cartografias quando usamos a **contagem** de homicídios do que a **taxa**. Isso diz algo sobre o fenômeno: a presença de facção está ligada ao **volume** absoluto de violência e de mercado (municípios maiores), não só à intensidade relativa à população. Com a taxa, o EM também separa um pequeno grupo de municípios **sem nenhum homicídio**; o piso de variância em `em.py` impede que essa componente colapse sobre o zero.

## 4. Naive Bayes puro × Naive Bayes com o regime Z do EM

In [8]:
def cv_nb_em(limiar=0.5):
    acc, bal, f1 = [], [], []
    for tr, te in cv.split(X, y):
        m_em = MisturaGaussianaEM(K=3).fit(x[tr])
        Xz = tercis_do_treino(d, X, tr); Xz["Z"] = m_em.predizer(x)
        Xz = Xz.drop(columns=["TXH"])
        p = NaiveBayesCategorico(limiar=limiar).fit(Xz.iloc[tr], y[tr]).predict(Xz.iloc[te])
        acc.append(accuracy_score(y[te], p)); bal.append(balanced_accuracy_score(y[te], p)); f1.append(f1_score(y[te], p))
    return np.mean(acc), np.mean(bal), np.mean(f1)

prior = y.mean()
linhas = []
for lim in (0.5, prior):
    linhas.append(["NB puro", lim, *cv_nb(X, lim)])
    linhas.append(["NB + EM (Z)", lim, *cv_nb_em(lim)])
pd.DataFrame(linhas, columns=["modelo", "limiar", "acurácia", "acur. balanceada", "F1"]).round(3)

,modelo,limiar,acurácia,acur. balanceada,F1
0,NB puro,0.500,0.772,0.737,0.642
1,NB + EM (Z),0.500,0.777,0.745,0.653
2,NB puro,0.321,0.752,0.751,0.658
3,NB + EM (Z),0.321,0.755,0.746,0.653


**Leitura.** Sem ver o rótulo, o EM encontra três grupos cuja composição acompanha os regimes do Cartografias: no grupo de menor violência, quase todos os municípios estão "sem registro"; no de maior violência, a maioria tem facção e é ali que se concentram as **disputas**. Os grupos se sobrepõem, porque domínio sem violência ("paz do domínio") e violência sem facção existem. Com a tolerância relativa de 10⁻⁴, o EM converge em poucas iterações; com tolerâncias menores, a log-verossimilhança continua subindo muito devagar até o limite de iterações.

Trocar os tercis fixos `TXH` (taxa) pelo regime Z (aprendido da contagem) melhora um pouco o Naive Bayes com limiar 0,5, porque Z traz o **volume** de violência, que a taxa não carrega.

## 5. Probabilidades para a Etapa 6 (fora da dobra)

As probabilidades desta etapa são reutilizadas no sistema fuzzy (Etapa 6). Para cada município gravamos, em `dados/probabilidades_etapa4.csv`:

- `p_nb_em`: P(facção) do **Naive Bayes com o regime Z do EM**;
- `p_regime_violento`: P(Z = 2 | homicídios), a responsabilidade do grupo mais violento do EM, onde se concentram as disputas.

As duas são calculadas **fora da dobra**: o EM e o Naive Bayes são ajustados nas outras 4 dobras, as mesmas das Etapas 1 e 5.

In [9]:
from sklearn.metrics import roc_auc_score
p_nb_em, p_z2 = np.zeros(len(d)), np.zeros(len(d))
for tr, te in cv.split(X, y):
    m_em = MisturaGaussianaEM(K=3).fit(x[tr])
    Xz = tercis_do_treino(d, X, tr); Xz["Z"] = m_em.predizer(x)
    Xz = Xz.drop(columns=["TXH"])
    p_nb_em[te] = NaiveBayesCategorico().fit(Xz.iloc[tr], y[tr]).predict_proba1(Xz.iloc[te])
    p_z2[te] = m_em.responsabilidades(x[te])[:, 2]
d["p_nb_em"], d["p_regime_violento"] = p_nb_em, p_z2
d[["cod_ibge", "uf", "municipio", "p_nb_em", "p_regime_violento"]].to_csv("dados/probabilidades_etapa4.csv", index=False)
print(f"AUC (facção 2024) fora da dobra: NB + EM = {roc_auc_score(y, p_nb_em):.3f}  |  P(regime violento) = {roc_auc_score(y, p_z2):.3f}")

AUC (facção 2024) fora da dobra: NB + EM = 0.840  |  P(regime violento) = 0.787
